# Exploration du schema des tables DP1 via TAP

- **Auteur** : Sylvie Dagoret-Campagne
- **Date** : 2026-02-28

## JOINs corrects (valides par tests)

| JOIN | Cle | Remarque |
|------|-----|----------|
| `Source` x `Visit` | `src.Visit = v.visit` | OK |
| `Source` x `CcdVisit` | `src.Visit = cv.VisitId` | OK - **sans** `AND detector` |
| `ForcedSource` x `CcdVisit` | `fs.Visit = cv.VisitId` | OK (exemple officiel) |
| `Object` x `ForcedSource` x `CcdVisit` | `obj.objectId = fs.objectId` puis `fs.Visit = cv.VisitId` | OK (exemple officiel) |

## Piege confirme
- `dp1.Source` **n'a PAS de colonne `detector`** -> le JOIN avec CcdVisit est 1-to-many (une source par visite, plusieurs CCDs par visite)
- La casse compte : `Source.Visit` (V majuscule), `CcdVisit.VisitId` (camelCase), `Visit.visit` (minuscule)

## Reference officielle
```sql
-- Source JOIN CcdVisit (doc dp1.lsst.io/tutorials/portal/103/portal-103-4.html)
SELECT src.sourceId, src.band, src.Visit, cv.VisitId, cv.expMidptMJD, cv.seeing
FROM dp1.Source AS src
JOIN dp1.CcdVisit AS cv ON src.Visit = cv.VisitId
WHERE CONTAINS(POINT('ICRS', src.coord_ra, src.coord_dec),
               CIRCLE('ICRS', 53.13, -28.10, 0.05)) = 1
AND src.band = 'i'
```

## 1. Connexion TAP

In [1]:
import os
import pandas as pd
import pyvo
import requests
from IPython.display import display

token = os.environ["RSP_TOKEN"]
session = requests.Session()
session.headers["Authorization"] = f"Bearer {token}"

RSP_TAP_URL = "https://data.lsst.cloud/api/tap"
service = pyvo.dal.TAPService(RSP_TAP_URL, session=session)
print("Connecte :", RSP_TAP_URL)

Connecte : https://data.lsst.cloud/api/tap


In [2]:
def show_schema(service, table_name):
    """Affiche le schema complet d'une table via tap_schema."""
    query = f"""
    SELECT column_name, datatype, description, unit
    FROM tap_schema.columns
    WHERE table_name = '{table_name}'
    ORDER BY column_name
    """
    df = service.search(query).to_table().to_pandas()
    print(f"\n{'='*70}")
    print(f"TABLE : {table_name}  ({len(df)} colonnes)")
    print(f"{'='*70}")
    if len(df) == 0:
        print("  !! Table non trouvee !!")
    else:
        pd.set_option('display.max_rows', None)
        pd.set_option('display.max_colwidth', 80)
        display(df)
    return df

print("OK")

OK


## 2. Liste de toutes les tables DP1

In [3]:
tables_result = service.search("""
    SELECT table_name, description
    FROM tap_schema.tables
    WHERE table_name LIKE 'dp1.%'
    ORDER BY table_name
""").to_table().to_pandas()
display(tables_result)

,table_name,description
0,dp1.CcdVisit,Metadata about the nine individual CCD images ...
1,dp1.CoaddPatches,Static information about the subset of tracts ...
2,dp1.DiaObject,Properties of time-varying astronomical object...
3,dp1.DiaSource,Properties of transient-object detections on t...
4,dp1.ForcedSource,Forced-photometry measurements on individual s...
5,dp1.ForcedSourceOnDiaObject,Point-source forced-photometry measurements on...
6,dp1.MPCORB,Orbit catalog produced by the Minor Planet Cen...
7,dp1.Object,Descriptions of static astronomical objects (o...
8,dp1.Source,Properties of detections on the single-epoch v...
9,dp1.SSObject,LSST-computed per-object quantities. 1:1 relat...


## 3. Schemas des tables principales

In [4]:
df_source = show_schema(service, 'dp1.Source')


TABLE : dp1.Source  (156 colonnes)


,column_name,datatype,description,unit
0,ap03Flux,double,Flux within 3.0-pixel aperture,nJy
1,ap03FluxErr,double,Flux uncertainty within 3.0-pixel aperture,nJy
2,ap03Flux_flag,boolean,General Failure Flag,
3,ap06Flux,double,Flux within 6.0-pixel aperture,nJy
4,ap06FluxErr,double,Flux uncertainty within 6.0-pixel aperture,nJy
5,ap06Flux_flag,boolean,General Failure Flag,
6,ap09Flux,double,Flux within 9.0-pixel aperture,nJy
7,ap09FluxErr,double,Flux uncertainty within 9.0-pixel aperture,nJy
8,ap09Flux_flag,boolean,General Failure Flag,
9,ap12Flux,double,Flux within 12.0-pixel aperture,nJy


In [5]:
df_visit = show_schema(service, 'dp1.Visit')


TABLE : dp1.Visit  (15 colonnes)


,column_name,datatype,description,unit
0,airmass,double,Airmass of the observed line of sight.,
1,altitude,double,Altitude of focal plane center at the middle of the visit.,deg
2,azimuth,double,Azimuth of focal plane center at the middle of the visit.,deg
3,band,char,Name of the band used to take the visit where this source was measured. Abst...,
4,dec,double,Declination of focal plane center,deg
5,expMidpt,char,Midpoint time for exposure at the fiducial center of the focal plane array. ...,
6,expMidptMJD,double,Midpoint time for exposure at the fiducial center of the focal plane array i...,d
7,expTime,double,"Spatially-averaged duration of visit, accurate to 10ms.",s
8,obsStart,char,"Start time of the visit at the fiducial center of the focal plane array, TAI...",
9,obsStartMJD,double,"Start of the exposure in MJD, TAI, accurate to 10ms.",d


In [6]:
df_ccdvisit = show_schema(service, 'dp1.CcdVisit')


TABLE : dp1.CcdVisit  (50 colonnes)


,column_name,datatype,description,unit
0,astromOffsetMean,double,Mean offset of astrometric calibration matches (arcsec),arcsec
1,astromOffsetStd,double,Standard deviation of offsets of astrometric calibration matches (arcsec),arcsec
2,band,char,Name of the band used to take the visit where this source was measured. Abst...,
3,ccdVisitId,long,Primary key (unique identifier).,
4,darkTime,double,"Average dark current accumulation time, accurate to 10ms.",s
5,dec,double,Declination of Ccd center.,deg
6,detector,long,Detector ID. A detector associated with a particular instrument (not an obse...,
7,effTime,double,Effective time metric,s
8,effTimePsfSigmaScale,double,Effective time metric -- PSF size component,
9,effTimeSkyBgScale,double,Effective time metric -- Sky background component,


In [7]:
df_forcedsource = show_schema(service, 'dp1.ForcedSource')


TABLE : dp1.ForcedSource  (28 colonnes)


,column_name,datatype,description,unit
0,band,char,Abstract filter that is not associated with a particular instrument,
1,coord_dec,double,Fiducial ICRS Declination of centroid used for database indexing,deg
2,coord_ra,double,Fiducial ICRS Right Ascension of centroid used for database indexing,deg
3,detector,long,Id of the detector where this source was measured.,
4,diff_PixelFlags_nodataCenter,boolean,Source center is outside usable region on image difference (masked NO_DATA),
5,invalidPsfFlag,boolean,Source has an invalid PSF.,
6,objectId,long,Unique Object ID. Primary Key of the Object Table,
7,parentObjectId,long,Unique ObjectId of the parent of the ObjectId in context of the deblender.,
8,patch,long,Skymap patch ID,
9,pixelFlags_bad,boolean,Bad pixel in the Source footprint,


In [8]:
df_diasource = show_schema(service, 'dp1.DiaSource')


TABLE : dp1.DiaSource  (87 colonnes)


,column_name,datatype,description,unit
0,apFlux,float,Flux in a 12 pixel radius aperture on the difference image.,nJy
1,apFluxErr,float,Estimated uncertainty of apFlux.,nJy
2,apFlux_flag,boolean,General aperture flux algorithm failure flag; set if anything went wrong whe...,
3,apFlux_flag_apertureTruncated,boolean,Aperture did not fit within measurement image.,
4,band,char,Band used to take this observation.,
5,bboxSize,long,Bounding box of diaSource footprint.,
6,centroid_flag,boolean,General centroid algorithm failure flag; set if anything went wrong when fit...,
7,coord_dec,double,Fiducial ICRS Declination of centroid used for database indexing.,deg
8,coord_ra,double,Fiducial ICRS Right Ascension of centroid used for database indexing.,deg
9,dec,double,Position in declination.,deg


In [9]:
df_object = show_schema(service, 'dp1.Object')


TABLE : dp1.Object  (1296 colonnes)


,column_name,datatype,description,unit
0,coord_dec,double,Fiducial ICRS Declination of centroid used for database indexing,deg
1,coord_decErr,float,Error in fiducial ICRS Declination of centroid,deg
2,coord_ra,double,Fiducial ICRS Right Ascension of centroid used for database indexing,deg
3,coord_ra_dec_Cov,float,Covariance between fiducial ICRS Right Ascension and Declination of centroid,deg**2
4,coord_raErr,float,Error in fiducial ICRS Right Ascension of centroid,deg
5,deblend_failed,boolean,Deblender failed to deblend this source,
6,deblend_incompleteData,boolean,One or more bands were not deblended due to an inability to model the PSF.,
7,deblend_isolatedParent,boolean,Deblender skipped this footprint because there was only a single peak,
8,deblend_iterations,int,Number of iterations during deblending,
9,deblend_logL,float,Log likelihood of the entire blend in scarlet_lite.,


In [10]:
df_diaobject = show_schema(service, 'dp1.DiaObject')


TABLE : dp1.DiaObject  (137 colonnes)


,column_name,datatype,description,unit
0,dec,double,Declination coordinate of the position of the diaObject at time radecMjdTai.,deg
1,diaObjectId,long,Unique identifier of this DiaObject.,
2,g_psfFluxChi2,float,Chi^2 statistic for the scatter of g_psfFlux around g_psfFluxMean,
3,g_psfFluxErrMean,float,Mean of the diaSource PSF flux errors,nJy
4,g_psfFluxLinearIntercept,double,y-intercept of a linear model fit to diaSource PSF flux vs time,nJy
5,g_psfFluxLinearSlope,double,Slope of a linear model fit to diaSource PSF flux vs time,nJy/d
6,g_psfFluxMAD,float,Median absolute deviation of diaSource PSF flux. Does not include scale fact...,nJy
7,g_psfFluxMax,double,Maximum diaSource PSF flux,nJy
8,g_psfFluxMaxSlope,double,Maximum ratio of time ordered deltaFlux / deltaTime,nJy/d
9,g_psfFluxMean,double,Weighted mean of diaSource PSF flux,nJy


## 4. Recap des colonnes de JOIN par table

In [11]:
tables = {
    'dp1.Source':       df_source,
    'dp1.Visit':        df_visit,
    'dp1.CcdVisit':     df_ccdvisit,
    'dp1.ForcedSource': df_forcedsource,
    'dp1.DiaSource':    df_diasource,
    'dp1.Object':       df_object,
    'dp1.DiaObject':    df_diaobject,
}

join_candidates = ['visit', 'Visit', 'visitId', 'VisitId', 'detector',
                   'ccdVisitId', 'objectId', 'diaObjectId', 'sourceId', 'diaSourceId']

print(f"{'Table':<25}  Colonnes de JOIN presentes")
print('-'*70)
for tname, tdf in tables.items():
    if tdf is not None and len(tdf) > 0:
        found = [c for c in join_candidates if c in tdf['column_name'].values]
        print(f"{tname:<25}  {found}")

Table                      Colonnes de JOIN presentes
----------------------------------------------------------------------
dp1.Source                 ['visit', 'detector', 'sourceId']
dp1.Visit                  ['visit']
dp1.CcdVisit               ['visitId', 'detector', 'ccdVisitId']
dp1.ForcedSource           ['visit', 'detector', 'objectId']
dp1.DiaSource              ['visit', 'detector', 'diaObjectId', 'diaSourceId']
dp1.Object                 ['objectId']
dp1.DiaObject              ['diaObjectId']


## 5. Tests de JOIN (requetes de validation)

D'apres les tests precedents et la doc officielle dp1.lsst.io :
- `Source` n'a **pas** de colonne `detector` -> JOIN avec CcdVisit sur `Visit = VisitId` uniquement
- Le JOIN est donc 1-to-many : plusieurs CCDs par visite

In [12]:
# Test 1 : Source JOIN CcdVisit
# Cle : src.Visit = cv.VisitId  (sans AND detector)
# Ref : dp1.lsst.io/tutorials/portal/103/portal-103-4.html
print("Test 1 : Source JOIN CcdVisit  (src.Visit = cv.VisitId, sans detector)")
try:
    res1 = service.search("""
        SELECT TOP 5
            src.sourceId, src.Visit, src.band, src.psfFlux,
            cv.VisitId, cv.expMidptMJD, cv.seeing, cv.zeroPoint, cv.magLim
        FROM dp1.Source AS src
        JOIN dp1.CcdVisit AS cv ON src.Visit = cv.VisitId
        WHERE CONTAINS(
            POINT('ICRS', src.coord_ra, src.coord_dec),
            CIRCLE('ICRS', 53.13, -28.10, 0.05)
        ) = 1
        AND src.band = 'g'
    """).to_table().to_pandas()
    print(f"  OK - {len(res1)} lignes")
    display(res1)
except Exception as e:
    print(f"  ERREUR : {e}")

Test 1 : Source JOIN CcdVisit  (src.Visit = cv.VisitId, sans detector)
  OK - 5 lignes


,sourceId,Visit,band,psfFlux,VisitId,expMidptMJD,seeing,zeroPoint,magLim
0,600320190329653091,2024110800266,g,1015.433222,2024110800266,60623.274265,1.183205,32.078899,24.7481
1,600320190329653091,2024110800266,g,1015.433222,2024110800266,60623.274265,1.187859,32.077099,24.7698
2,600320190329653091,2024110800266,g,1015.433222,2024110800266,60623.274265,1.191796,32.076302,24.7442
3,600320190329653091,2024110800266,g,1015.433222,2024110800266,60623.274265,1.216140,32.075901,24.7030
4,600320190329653091,2024110800266,g,1015.433222,2024110800266,60623.274265,1.188101,32.078602,24.7570


In [13]:
# Test 2 : Source JOIN Visit JOIN CcdVisit (triple)
print("Test 2 : Source JOIN Visit JOIN CcdVisit (triple)")
try:
    res2 = service.search("""
        SELECT TOP 5
            src.sourceId, src.Visit, src.band,
            src.psfFlux, src.psfFluxErr,
            v.expMidptMJD, v.airmass,
            cv.VisitId, cv.seeing, cv.zeroPoint, cv.magLim, cv.skyBg, cv.skyNoise
        FROM dp1.Source AS src
        JOIN dp1.Visit    AS v  ON src.Visit = v.visit
        JOIN dp1.CcdVisit AS cv ON src.Visit = cv.VisitId
        WHERE CONTAINS(
            POINT('ICRS', src.coord_ra, src.coord_dec),
            CIRCLE('ICRS', 53.13, -28.10, 0.05)
        ) = 1
        AND src.band = 'g'
    """).to_table().to_pandas()
    print(f"  OK - {len(res2)} lignes")
    display(res2)
except Exception as e:
    print(f"  ERREUR : {e}")

Test 2 : Source JOIN Visit JOIN CcdVisit (triple)
  OK - 5 lignes


,sourceId,Visit,band,psfFlux,psfFluxErr,expMidptMJD,airmass,VisitId,seeing,zeroPoint,magLim,skyBg,skyNoise
0,600320190329653091,2024110800266,g,1015.433222,115.385564,60623.274265,1.065913,2024110800266,1.183205,32.078899,24.7481,435.773987,24.916901
1,600320190329653091,2024110800266,g,1015.433222,115.385564,60623.274265,1.065913,2024110800266,1.187859,32.077099,24.7698,440.821991,21.669800
2,600320190329653091,2024110800266,g,1015.433222,115.385564,60623.274265,1.065913,2024110800266,1.191796,32.076302,24.7442,440.308990,24.076200
3,600320190329653091,2024110800266,g,1015.433222,115.385564,60623.274265,1.065913,2024110800266,1.216140,32.075901,24.7030,434.510986,23.580099
4,600320190329653091,2024110800266,g,1015.433222,115.385564,60623.274265,1.065913,2024110800266,1.188101,32.078602,24.7570,438.098999,22.790199


In [14]:
# Test 3 : Object JOIN ForcedSource JOIN CcdVisit (exemple officiel - deja valide)
print("Test 3 : Object JOIN ForcedSource JOIN CcdVisit (exemple officiel)")
try:
    res3 = service.search("""
        SELECT TOP 5
            obj.objectId, obj.refExtendedness,
            scisql_nanojanskyToAbMag(fs.psfFlux) AS fs_psfAbMag,
            cv.VisitId, cv.expMidptMJD, cv.seeing
        FROM dp1.Object AS obj
        JOIN dp1.ForcedSource AS fs ON obj.objectId = fs.objectId
        JOIN dp1.CcdVisit     AS cv ON fs.Visit     = cv.VisitId
        WHERE CONTAINS(
            POINT('ICRS', obj.coord_ra, obj.coord_dec),
            CIRCLE('ICRS', 53.13, -28.10, 0.05)
        ) = 1
        AND obj.refExtendedness = 1
        AND fs.band = 'g'
    """).to_table().to_pandas()
    print(f"  OK - {len(res3)} lignes")
    display(res3)
except Exception as e:
    print(f"  ERREUR : {e}")

Test 3 : Object JOIN ForcedSource JOIN CcdVisit (exemple officiel)
  OK - 5 lignes


,objectId,refExtendedness,fs_psfAbMag,VisitId,expMidptMJD,seeing
0,611254385447554248,1.0,25.517561,2024120600078,60651.094041,1.244865
1,611254385447554248,1.0,25.517561,2024120600078,60651.094041,1.187256
2,611254385447554248,1.0,25.517561,2024120600078,60651.094041,1.152859
3,611254385447554248,1.0,25.517561,2024120600078,60651.094041,1.182875
4,611254385447554248,1.0,25.517561,2024120600078,60651.094041,1.135589


In [15]:
# Test 4 : DiaSource JOIN Visit JOIN CcdVisit
# On teste les deux casses possibles pour dia.visit
print("Test 4 : DiaSource JOIN Visit JOIN CcdVisit")
for visit_col in ['visit', 'Visit']:
    try:
        res4 = service.search(f"""
            SELECT TOP 3
                dia.diaSourceId, dia.{visit_col}, dia.band, dia.psfFlux,
                v.expMidptMJD, v.airmass,
                cv.seeing, cv.zeroPoint
            FROM dp1.DiaSource AS dia
            JOIN dp1.Visit    AS v  ON dia.{visit_col} = v.visit
            JOIN dp1.CcdVisit AS cv ON dia.{visit_col} = cv.VisitId
            WHERE CONTAINS(
                POINT('ICRS', dia.coord_ra, dia.coord_dec),
                CIRCLE('ICRS', 53.13, -28.10, 0.05)
            ) = 1
            AND dia.band = 'g'
        """).to_table().to_pandas()
        print(f"  OK avec dia.{visit_col} - {len(res4)} lignes")
        display(res4)
        break
    except Exception as e:
        print(f"  ERREUR avec dia.{visit_col} : {e}")

Test 4 : DiaSource JOIN Visit JOIN CcdVisit
  OK avec dia.visit - 3 lignes


,diaSourceId,visit,band,psfFlux,expMidptMJD,airmass,seeing,zeroPoint
0,600320192611876868,2024110800283,g,24915.199219,60623.292555,1.112068,1.169730,32.069000
1,600320192611876868,2024110800283,g,24915.199219,60623.292555,1.112068,1.128123,32.071400
2,600320192611876868,2024110800283,g,24915.199219,60623.292555,1.112068,1.133330,32.069302


## 6. Requete principale pour l'analyse de saturation Gaia

Version finale corrigee : Source JOIN Visit JOIN CcdVisit sans colonne `detector` dans Source.

In [16]:
# Parametres
RA_CENTER  = 53.0
DEC_CENTER = -28.0
RADIUS_DEG = 1.75
LSST_MAG_LIMIT = 20.5
FLUX_LIMIT = 10**(-0.4 * (LSST_MAG_LIMIT - 31.4))

query_final = f"""
SELECT
    src.sourceId,
    src.Visit,
    src.band,
    src.coord_ra  AS ra,
    src.coord_dec AS dec,
    src.psfFlux,
    src.psfFluxErr,
    v.expMidptMJD,
    v.airmass,
    cv.VisitId,
    cv.seeing,
    cv.zeroPoint,
    cv.magLim,
    cv.skyBg,
    cv.skyNoise
FROM dp1.Source AS src
JOIN dp1.Visit    AS v  ON src.Visit = v.visit
JOIN dp1.CcdVisit AS cv ON src.Visit = cv.VisitId
WHERE CONTAINS(
    POINT('ICRS', src.coord_ra, src.coord_dec),
    CIRCLE('ICRS', {RA_CENTER}, {DEC_CENTER}, {RADIUS_DEG})
) = 1
AND src.psfFlux > {FLUX_LIMIT}
AND src.band = 'g'
"""

print("Requete finale (Source JOIN Visit JOIN CcdVisit) :")
print(query_final)

Requete finale (Source JOIN Visit JOIN CcdVisit) :

SELECT
    src.sourceId,
    src.Visit,
    src.band,
    src.coord_ra  AS ra,
    src.coord_dec AS dec,
    src.psfFlux,
    src.psfFluxErr,
    v.expMidptMJD,
    v.airmass,
    cv.VisitId,
    cv.seeing,
    cv.zeroPoint,
    cv.magLim,
    cv.skyBg,
    cv.skyNoise
FROM dp1.Source AS src
JOIN dp1.Visit    AS v  ON src.Visit = v.visit
JOIN dp1.CcdVisit AS cv ON src.Visit = cv.VisitId
WHERE CONTAINS(
    POINT('ICRS', src.coord_ra, src.coord_dec),
    CIRCLE('ICRS', 53.0, -28.0, 1.75)
) = 1
AND src.psfFlux > 22908.6765276777
AND src.band = 'g'



In [17]:
# Execution asynchrone (necessaire pour les grandes requetes)
print("Soumission du job asynchrone...")
job = service.submit_job(query_final)
job.run()
job.wait(phases=["COMPLETED", "ERROR"])
print("Job phase:", job.phase)
if job.phase == "ERROR":
    job.raise_if_error()

df = job.fetch_result().to_table().to_pandas()
print(f"Resultat : {len(df)} lignes, {len(df.columns)} colonnes")
print("Colonnes :", df.columns.tolist())
df.head(3)

Soumission du job asynchrone...
Job phase: COMPLETED
Resultat : 1604205 lignes, 15 colonnes
Colonnes : ['sourceId', 'Visit', 'band', 'ra', 'dec', 'psfFlux', 'psfFluxErr', 'expMidptMJD', 'airmass', 'VisitId', 'seeing', 'zeroPoint', 'magLim', 'skyBg', 'skyNoise']


,sourceId,Visit,band,ra,dec,psfFlux,psfFluxErr,expMidptMJD,airmass,VisitId,seeing,zeroPoint,magLim,skyBg,skyNoise
0,600412549304811753,2024112900266,g,52.723334,-28.612956,71491.831847,288.542128,60644.257691,1.19374,2024112900266,1.108639,32.040401,24.6693,500.583008,26.283800
1,600412549304811753,2024112900266,g,52.723334,-28.612956,71491.831847,288.542128,60644.257691,1.19374,2024112900266,1.128576,32.040798,24.6980,505.597992,23.002001
2,600412549304811753,2024112900266,g,52.723334,-28.612956,71491.831847,288.542128,60644.257691,1.19374,2024112900266,1.155142,32.039700,24.6584,504.859009,25.265699


In [18]:
len(df)

1604205

In [19]:
df.to_csv("merged_sourcevisitvisitCCD.csv")